In [15]:
import argparse
from enum import Enum
from typing import Iterator, List

import os
import cv2
import numpy as np
import supervision as sv
from tqdm import tqdm
from ultralytics import YOLO

from sports.common.team import TeamClassifier

PARENT_DIR = os.path.abspath('')
PARENT_DIR = os.getcwd()

PLAYER_DETECTION_MODEL_PATH = os.path.join(PARENT_DIR, 'data/nba_detection.pt')
KEYPOINT_DETECTION_MODEL_PATH = os.path.join(PARENT_DIR, 'data/keypoint2_weights.pt')

PLAYER_CLASS_ID = 1

STRIDE = 60

COLORS = ['#FF1493', '#00BFFF', '#FF6347', '#FFD700']

BOX_ANNOTATOR = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(COLORS),
    thickness=2
)
ELLIPSE_ANNOTATOR = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(COLORS),
    thickness=2
)
BOX_LABEL_ANNOTATOR = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(COLORS),
    text_color=sv.Color.from_hex('#FFFFFF'),
    text_padding=5,
    text_thickness=1,
)
ELLIPSE_LABEL_ANNOTATOR = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(COLORS),
    text_color=sv.Color.from_hex('#FFFFFF'),
    text_padding=5,
    text_thickness=1,
    text_position=sv.Position.BOTTOM_CENTER,
)

class Mode(Enum):
    """
    Enum class representing different modes of operation for Soccer AI video analysis.
    """
    PITCH_DETECTION = 'PITCH_DETECTION'
    PLAYER_DETECTION = 'PLAYER_DETECTION'
    BALL_DETECTION = 'BALL_DETECTION'
    PLAYER_TRACKING = 'PLAYER_TRACKING'
    TEAM_CLASSIFICATION = 'TEAM_CLASSIFICATION'
    RADAR = 'RADAR'

def get_crops(frame: np.ndarray, detections: sv.Detections) -> List[np.ndarray]:
    """
    Extract crops from the frame based on detected bounding boxes.

    Args:
        frame (np.ndarray): The frame from which to extract crops.
        detections (sv.Detections): Detected objects with bounding boxes.

    Returns:
        List[np.ndarray]: List of cropped images.
    """
    return [sv.crop_image(frame, xyxy) for xyxy in detections.xyxy]


In [6]:
import warnings
warnings.filterwarnings("ignore", message="'force_all_finite' was renamed to 'ensure_all_finite'")

In [3]:
source_video_path = "./test_film.mp4"
device = 'mps'

In [17]:
player_detection_model = YOLO(PLAYER_DETECTION_MODEL_PATH).to(device=device)
keypoint_detection_model = YOLO(KEYPOINT_DETECTION_MODEL_PATH).to(device=device)


frame_generator = sv.get_video_frames_generator(
    source_path=source_video_path, stride=STRIDE)

crops = []
for frame in tqdm(frame_generator, desc='collecting crops'):
    result = player_detection_model(frame, imgsz=1280, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    crops += get_crops(frame, detections[detections.class_id == PLAYER_CLASS_ID])

team_classifier = TeamClassifier(device=device)
team_classifier.fit(crops)


collecting crops: 33it [00:03, 10.33it/s]


KeyboardInterrupt: 

In [ ]:
frame_generator = sv.get_video_frames_generator(source_path=source_video_path)
tracker = sv.ByteTrack(minimum_consecutive_frames=3)

keypoint_detection_model = YOLO(KEYPOINT_DETECTION_MODEL_PATH).to(device=device)
for frame in frame_generator:
    result = keypoint_detection_model(frame, imgsz=1280, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    detections = detections[detections.class_id == 2]

    if len(detections) > 0 and detections.mask is not None:
        # Visualize the frame with masks
        annotated_frame = frame.copy()
        for i, mask in enumerate(detections.mask):
            # Convert mask to uint8 format for contour detection
            binary_mask = mask.astype(np.uint8) * 255
            
            # Find contours of the mask
            contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if contours:
                # Get the largest contour
                largest_contour = max(contours, key=cv2.contourArea)
                
                # Approximate the contour to get polygon corners
                epsilon = 0.02 * cv2.arcLength(largest_contour, True)
                approx_polygon = cv2.approxPolyDP(largest_contour, epsilon, True)
                
                # Draw the polygon corners
                for point in approx_polygon:
                    cv2.circle(annotated_frame, (point[0][0], point[0][1]), 5, (0, 255, 0), -1)
                
                # Draw the polygon outline
                cv2.polylines(annotated_frame, [approx_polygon], True, (0, 255, 0), 2)
                
                # Print the corners coordinates
                print(f"Mask {i} corners: {approx_polygon.reshape(-1, 2)}")

    print(detections)
    
    
    # detections = tracker.update_with_detections(detections)

    

    # players = detections[detections.class_id == PLAYER_CLASS_ID]
    # crops = get_crops(frame, players)

    # players_team_id = team_classifier.predict(crops)

    # # detections = sv.Detections.merge([players, goalkeepers, referees])
    # color_lookup = np.array(
    #         players_team_id.tolist()
    # )
    # labels = [str(tracker_id) for tracker_id in players.tracker_id]

    # annotated_frame = frame.copy()
    # # try:
    # annotated_frame = ELLIPSE_ANNOTATOR.annotate(
    #     annotated_frame, players, custom_color_lookup=color_lookup)
    # # except Exception as e:
    # #     print("Error:", e)
    # #     print("team_id")
    # #     print(players_team_id)
    # #     print(f"# of crops {len(crops)}")
    # #     print("detections")

    # #     print(players)
    # #     break

    # annotated_frame = ELLIPSE_LABEL_ANNOTATOR.annotate(
    #     annotated_frame, players, labels, custom_color_lookup=color_lookup)

Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=array([], shape=(0, 720, 1280), dtype=bool), confidence=array([], dtype=float32), class_id=array([], dtype=int64), tracker_id=None, data={'class_name': array([], dtype='<U5')}, metadata={})
Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=array([], shape=(0, 720, 1280), dtype=bool), confidence=array([], dtype=float32), class_id=array([], dtype=int64), tracker_id=None, data={'class_name': array([], dtype='<U5')}, metadata={})
Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=array([], shape=(0, 720, 1280), dtype=bool), confidence=array([], dtype=float32), class_id=array([], dtype=int64), tracker_id=None, data={'class_name': array([], dtype='<U5')}, metadata={})
Mask 0 corners: [[  2 417]
 [  2 583]
 [649 601]
 [649 251]
 [305 226]]
Detections(xyxy=array([[     1.1542,      226.33,      647.38,      599.19]], dtype=float32), mask=array([[[False, False, False, ..., False, False, False],
        [False, Fal

KeyboardInterrupt: 

: 